# v3.5 link C — the head-crop arms

Six arms over the **51 pairs either failure record marks hard** (`v35_linkC.csv`), seeds **46/47/48**, self-hosted klein. Everything runs here — crops, parsing, both klein calls, the upscale. No step on a laptop between sessions.

| arm | call 1 | head off | ankle cut | computed here |
|---|---|---|---|---|
| `VEi` | the lock's `Q3` — mannequin head **and** re-pose | no | no | no — reused |
| `BC` | v3.1's incumbent: bald pass → V2 crop | the V2 cropper | no | no — reused |
| `VEic` | the lock's `Q3` | a crop | no | yes |
| `M1qbc` | `Q3` minus the mannequin sentence, **plus bald** | a crop | no | yes |
| `VEica` | as `VEic` | a crop | **yes** | yes |
| `M1qbca` | as `M1qbc` | a crop | **yes** | yes |

An arm name is read, not looked up: base (`VEi` | `M1qb`) + `c` head crop + `a` ankle cut.

**Why the re-pose arm is bald.** `BC` bald-passes the raw photograph before it crops, because hair on the shoulders and chest cannot be told from garment by any matte; `VEi` gets that free, since the mannequin sentence takes the head and its hair together. A re-pose arm that keeps the wearer's own head does neither. So `M1q` without the bald clause is not shippable and is not run — one klein call does both jobs, measured on the five longest-haired garments of the fold (`v3/report/v35_bald.html`).

**Order of operations, which is not the obvious one:**

    call 1 ──▶ white-margin re-crop ──▶ HEAD CROP ──▶ [ankle cut] ──▶ SR to ~1 MP ──▶ call 2

The crops go **before** the SR pass. Cropping afterwards would take the reference back below 1 MP and break the rule v3.4 link H bought — what conditioning contributes is bounded by its **token footprint** in call 2. Neither crop is new code: the head crop is `ironman_bc_crop.crop_bc`, the call that makes the `BC` references; the ankle cut is `run_ironman.ankle_cut`, v3.3's, reopened here as its own variable so each `a` arm is paired with its cut-less twin.

**Where things run.** klein and the SR pass are always on the GPU. The two ONNX models — BiRefNet and the human parser — default to **CPU**, deliberately: every reference of record, including the `BC` refs this comparison is measured against, was made with CPU inference, so CPU crops are numerically identical to the archive and cell 7's validation cannot drift. Set `CROP_DEVICE = 'gpu'` for ~2 min of crops instead of ~12, and cell 3 will find an onnxruntime whose CUDA provider actually loads.

**One session.** Needs on Drive under `v3_runs/`: `v34_ironman2_*.zip`, `v34_ironman2_bc_*.zip`, and optionally a previous `v35_linkC_*.zip`.

In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 0.689     # CAD/h at 5.3 CU/h x CAD 0.13/CU; edit if your rate differs
SEEDS = [46, 47, 48]          # the iron-man 2 seeds: every lock cell pairs with a verdict
ARMS = ("VEic", "M1qbc", "VEica", "M1qbca")
MATRIX = "v35_linkC.csv"      # 51 pairs: the union of the v3.4 lock's and v3.3's failures

# Where BiRefNet and the human parser run. 'cpu' is the default on purpose: the references
# of record were made on CPU, so CPU crops are numerically identical to them. 'gpu' is
# faster (~2 min vs ~12) and proves the pipeline is GPU-runnable end to end.
CROP_DEVICE  = 'cpu'          # 'cpu' | 'gpu'
CROP_WORKERS = 4              # CPU only: ORT and MediaPipe release the GIL, so threads work
DRIVE_PROJECT_DIR = "Side projects and shi"

In [ ]:
# 2 · Drive, the HF cache that holds klein, and where the crops run
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'; BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
assert os.path.isdir(BASE), f'Drive project dir not found: {BASE}'
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
cands = [os.path.join(MYDRIVE, 'hf_cache'), os.path.join(BASE, 'tryon_models', 'hf_cache'),
         os.path.join(BASE, 'hf_cache')]
found = [c for c in cands if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
os.environ['HF_HOME'] = found[0] if found else cands[0]
os.environ['V3_MODEL_DIR'] = os.path.join(BASE, 'v3_models')
os.environ['V2_ORT_GPU'] = '1' if CROP_DEVICE == 'gpu' else '0'
for d in (os.environ['HF_HOME'], os.environ['V3_MODEL_DIR']): os.makedirs(d, exist_ok=True)
print('HF_HOME     ', os.environ['HF_HOME'], '(klein cached)' if found else '(klein downloads, ~9 GB)')
print('V3_MODEL_DIR', os.environ['V3_MODEL_DIR'])
print('crops on    ', CROP_DEVICE.upper(), '- klein and SR are on the GPU either way')

In [ ]:
# 3 · install, then the bundle
#
# onnxruntime is the one fiddly dependency. The CPU wheel and the GPU wheel cannot coexist,
# and the newest GPU wheel is built against CUDA 13 while Colab's torch ships CUDA 12.x -
# so it registers CUDAExecutionProvider, fails to dlopen it at session creation, and falls
# back to CPU with no error. get_available_providers() cannot tell you that; only loading
# the provider library can. So: install the CPU wheel when crops are on CPU, and when they
# are not, walk candidate GPU builds until one's provider library actually loads.
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe opencv-contrib-python-headless
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1

import subprocess, sys
ORT_VERSION = None
if CROP_DEVICE == 'cpu':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'onnxruntime'], check=True)
    print('onnxruntime (CPU wheel) installed - crops run on CPU by choice')
else:
    open('/content/ort_probe.py', 'w').write('''
import ctypes, glob, os, site, sys
import onnxruntime as ort
dirs = sorted({d for p in site.getsitepackages() for d in glob.glob(os.path.join(p, "nvidia", "*", "lib"))})
os.environ["LD_LIBRARY_PATH"] = ":".join(dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
for d in dirs:
    for f in os.listdir(d):
        if ".so" in f and any(k in f for k in ("cudart", "cublas", "cudnn", "cufft", "curand")):
            try: ctypes.CDLL(os.path.join(d, f), mode=ctypes.RTLD_GLOBAL)
            except OSError: pass
so = glob.glob(os.path.dirname(ort.__file__) + "/capi/libonnxruntime_providers_cuda.so")
if not so:
    print("NO_CUDA_PROVIDER_IN_WHEEL"); sys.exit(1)
ctypes.CDLL(so[0], mode=ctypes.RTLD_GLOBAL)
print("OK", ort.__version__)
''')
    for spec in ['', '==1.22.0', '==1.21.1', '==1.20.1', '==1.19.2', '==1.18.1']:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'onnxruntime-gpu{spec}'],
                           capture_output=True, text=True)
        if r.returncode:
            print(f'  onnxruntime-gpu{spec or " (newest)"}: no installable wheel'); continue
        t = subprocess.run([sys.executable, '/content/ort_probe.py'], capture_output=True, text=True)
        line = ((t.stdout + t.stderr).strip().splitlines() or [''])[-1][:130]
        print(f'  onnxruntime-gpu{spec or " (newest)"}: '
              f'{"CUDA OK -> " + line if not t.returncode else "CPU only (" + line + ")"}')
        if not t.returncode:
            ORT_VERSION = line.split()[-1]; break
    if not ORT_VERSION:
        print('no onnxruntime build here loads CUDA - crops will fall back to CPU')

!cd /content && rm -rf v35 && wget -q -O v35_bundle.zip https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v35_linkC_bundle.zip && unzip -qo v35_bundle.zip -d v35
%cd /content/v35
import torch, onnxruntime as ort
for f in ('realesr-general-x4v3.pth', 'lib/run_v35_linkC.py', 'lib/ironman_bc_crop.py',
          'lib/phase3_variants.py', 'lib/garment_crop.py', 'v35_linkC.csv'):
    assert os.path.exists(f), f'bundle incomplete: {f}'
os.makedirs('run/inputs', exist_ok=True)
assert torch.cuda.is_available(), 'no GPU - Runtime > Change runtime type > A100'
print(f'\n{torch.cuda.get_device_name(0)} | torch CUDA {torch.version.cuda} | onnxruntime {ort.__version__}')

In [ ]:
# 4 · everything already computed, off Drive. Nothing here is recomputed: these are the
#     records this arm is measured against, and redrawing them would break the pairing
#     with the human verdicts.
import glob, zipfile as zf, csv
rows = list(csv.DictReader(open(MATRIX)))
pairs = {r['set_id'] for r in rows}
stems = {r['person'] for r in rows} | {r['garment'] for r in rows}
garments = {r['garment'] for r in rows}
BASES = sorted({a.rstrip('ca') for a in ARMS})
WANT = ({f'inputs/{s}.jpg' for s in stems}
        | {f'inputs/{g}__A4.jpg' for g in garments}
        | {f'refs/{g}__{t}.jpg' for g in garments
           for t in ('VEi_small', 'VEi', 'BC', 'bald', 'VEi_headcut', 'M1qb_small',
                     'M1qb_headcut', 'VEic', 'VEica', 'M1qbc', 'M1qbca')}
        | {f'gen/{sid}__{a}__s{s}.jpg' for sid in pairs for s in SEEDS
           for a in ('VEi', 'BC', 'VEic', 'VEica', 'M1qbc', 'M1qbca')})

def pick(pat, exclude=None, required=True):
    zs = [z for z in sorted(glob.glob(os.path.join(BASE, 'v3_runs', pat)))
          if not (exclude and exclude in os.path.basename(z))]
    assert zs or not required, f'no {pat} on Drive under v3_runs/'
    return zs[-1] if zs else None

# '_bc_' sorts AFTER the digits, so a bare v34_ironman2_*.zip glob resolves to the BC zip
zips = [z for z in (pick('v34_ironman2_*.zip', exclude='_bc_'),
                    pick('v34_ironman2_bc_*.zip'),
                    pick('v35_linkC_*.zip', required=False)) if z]
assert len(set(zips)) == len(zips), f'two patterns resolved to the same file: {zips}'
got = 0
for zp in zips:
    with zf.ZipFile(zp) as z:
        members = [n for n in z.namelist() if n in WANT]
        z.extractall('run', members=members); got += len(members)
    print(f'  {os.path.basename(zp)}: {len(members)} files')

n_small = len(glob.glob('run/refs/*__VEi_small.jpg')); n_bc = len(glob.glob('run/refs/*__BC.jpg'))
base_cells = len(glob.glob('run/gen/*__VEi__*.jpg')) + len(glob.glob('run/gen/*__BC__*.jpg'))
print(f'{got} files | VEi_small {n_small}/{len(garments)} · BC refs {n_bc}/{len(garments)} · '
      f'baseline cells {base_cells}/{2 * len(rows) * len(SEEDS)}')
todo_refs = sum(not os.path.exists(f'run/refs/{g}__{b}_small.jpg') for g in garments for b in BASES)
todo_crops = sum(not os.path.exists(f'run/refs/{g}__{b}_headcut.jpg') for g in garments for b in BASES)
todo_edits = sum(not os.path.exists(f"run/gen/{r['set_id']}__{a}__s{s}.jpg")
                 for r in rows for a in ARMS for s in SEEDS)
print(f'to compute: {todo_refs} references + {todo_crops} head crops + {todo_edits} edits')
assert n_small == len(garments), "the lock's pre-SR references are incomplete"

In [ ]:
# 5 · load klein once, timed
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 6 · where every stage actually runs, measured on this hardware
import time, numpy as np, cv2, torch
import v3lib as L, run_ironman as R, garment_crop as GC, phase3_variants as P
paths = L.fetch_models(persist=os.environ.get('V3_MODEL_DIR'))
probe_g = sorted(garments)[0]
img = cv2.imread(f'run/inputs/{probe_g}.jpg')

t = time.time(); L.crop_a4(img, paths);              t_a4 = time.time() - t
t = time.time(); GC.biref_matte(img, 'audit', True); t_v2 = time.time() - t
t = time.time(); P.parse_human(img);                 t_ps = time.time() - t
t = time.time(); R.to_1mp_sr(cv2.resize(img, (320, 480))); t_sr = time.time() - t

audit = [('klein (torch)',             K.info().get('gpu', '?'),        None),
         ('SR upscaler (torch)',       R._SR.get('dev', '?'),           t_sr),
         ('BiRefNet - A4 crop',        L._S.get('biref_prov', '?'),     t_a4),
         ('BiRefNet - head crop (V2)', GC._STATE.get('biref_prov', '?'), t_v2),
         ('human parser (V2)',         P._HP['m'].get_providers()[0] if P._HP.get('m') else 'unavailable', t_ps)]
print(f"{'stage':28s} {'device / provider':32s} seconds")
for name, dev, secs in audit:
    print(f"{name:28s} {str(dev):32s} {'' if secs is None else f'{secs:6.2f}'}")

on_gpu = lambda d: any(t in str(d) for t in ('CUDA', 'cuda', 'NVIDIA'))
# klein and SR MUST be on the GPU. The ONNX crops are a choice, reported not enforced.
must = [n for n, d, _ in audit if not on_gpu(d) and ('klein' in n or 'SR' in n)]
assert not must, f'the generative stages are not on the GPU: {must}'
cpu_stages = [n for n, d, _ in audit if not on_gpu(d)]
if CROP_DEVICE == 'gpu' and cpu_stages:
    print(f'\nWARNING - asked for GPU crops, these fell back: {cpu_stages}')
if cpu_stages:
    n = len(garments) * len(BASES)
    print(f'\ncrops on CPU by {"fallback" if CROP_DEVICE == "gpu" else "choice"}: '
          f'{n} at ~{t_v2 + t_ps:.0f}s = ~{n * (t_v2 + t_ps) / 60:.0f} min sequential, '
          f'~{n * (t_v2 + t_ps) / 60 / CROP_WORKERS:.0f} min on {CROP_WORKERS} threads')
else:
    print('\nall five stages on the GPU')

In [ ]:
# 7 · the cropper, validated against the refs of record - then the case that has never
#     been proved: does the head come off a generated MANNEQUIN head? BC only ever cropped
#     a bald photograph, never one klein drew.
import glob
import ironman_bc_crop as C
from IPython.display import display
from PIL import Image

def side_by_side(imgs, h=440):
    fit = [cv2.resize(i, (max(1, int(i.shape[1] * h / i.shape[0])), h)) for i in imgs if i is not None]
    w = max(f.shape[1] for f in fit)
    return Image.fromarray(cv2.cvtColor(np.hstack(
        [np.pad(f, ((0, 0), (0, w - f.shape[1]), (0, 0)), constant_values=255) for f in fit]),
        cv2.COLOR_BGR2RGB))

bad = 0
for vp in sorted(glob.glob('validation/*__BC.jpg')):
    stem = os.path.basename(vp)[:-len('__BC.jpg')]
    src = f'run/refs/{stem}__bald.jpg'
    if not os.path.exists(src):
        continue
    a = cv2.imread(vp); b, _ = C.crop_bc(cv2.imread(src), f'val_{stem}')
    if abs(a.shape[0] - b.shape[0]) > 8 or abs(a.shape[1] - b.shape[1]) > 8:
        print(f'  validate {stem}: SHAPE {a.shape[:2]} vs {b.shape[:2]}'); bad += 1; continue
    mad = float(np.abs(a.astype(np.float32) - cv2.resize(b, (a.shape[1], a.shape[0])).astype(np.float32)).mean())
    print(f'  validate {stem}: MAD {mad:.2f}')
    bad += mad > 4.0
assert not bad, 'the cropper does not reproduce the refs of record on this machine'

small = cv2.imread(f'run/refs/{probe_g}__VEi_small.jpg')
t0 = time.time(); cut, cranium = C.crop_bc(small, f'probe_{probe_g}'); secs = time.time() - t0
print(f'\n{probe_g}: {small.shape[1]}x{small.shape[0]} -> {cut.shape[1]}x{cut.shape[0]}  '
      f'cranium_used={cranium}  {secs:.1f}s')
display(side_by_side([cv2.imread(f'run/inputs/{probe_g}__A4.jpg'), small, cut]))
assert cranium, 'the parser did not fire on the mannequin head - look before running the arm'

In [ ]:
# 8 · one pair end to end, before paying for the set
import run_v35_linkC as V
V.main(MATRIX, 'testset', seeds=SEEDS[:1], arms=ARMS,
       gpu_usd_per_hour=A100_USD_PER_HOUR, crop_workers=CROP_WORKERS, limit=1)
print(sorted(f for f in os.listdir('run/gen') if any(f'__{a}__' in f for a in ARMS))[:8])

In [ ]:
# 9 · the run (resumable - rerun after any disconnect and it skips what is on disk)
import run_v35_linkC as V, json
V.main(MATRIX, 'testset', seeds=SEEDS, arms=ARMS,
       gpu_usd_per_hour=A100_USD_PER_HOUR, crop_workers=CROP_WORKERS)
print(json.dumps(json.load(open('run/meta/cost_v35.json')), indent=1))

In [ ]:
# 10 · every cell landed? the parser fired? where was the ankle cut a no-op?
import json, glob
meta = json.load(open('run/meta/prompts_v35.json'))
want = len(rows) * len(SEEDS)
missed = sorted({g for g in garments for b in BASES if not meta.get(g, {}).get(f'{b}_cranium_used')})
print('references where the parser did NOT fire:', missed or 'none')
cut_refs = [(g, a) for g in garments for a in ARMS if a.endswith('a')]
noop = [(g, a) for g, a in cut_refs if meta.get(g, {}).get(f'{a}_ankle_row') is None]
print(f'ankle cut a no-op (no ankles in frame) on {len(noop)} of {len(cut_refs)} cut references')
short = []
for a in list(ARMS) + ['VEi', 'BC']:
    n = len(glob.glob(f'run/gen/*__{a}__*.jpg'))
    print(f'  {a:7s} {n}/{want} cells')
    if n < want: short.append(a)
assert not short, f'incomplete arms: {short} - rerun cell 9, it resumes'
print(f'\ncomplete: {(len(ARMS) + 2) * want} cells across {len(ARMS) + 2} arms')
if missed:
    print(f'NOTE: {len(missed)} garment(s) fell back off the parser; their head crops are '
          'flagged in the meta and read separately on the page, not averaged in.')

In [ ]:
# 11 · zip references, outputs and meta to Drive - verified before it leaves the session
import shutil, time, zipfile
name = f"v35_linkC_{time.strftime('%Y%m%d_%H%M')}"
KEEP = tuple(f'__{a}.' for a in ARMS) + ('__VEi_small', '__M1qb_small', '__VEi_headcut',
                                         '__M1qb_headcut', '__BC.', '__VEi.')
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    kept = [f for f in os.listdir('run/refs') if any(t in f for t in KEEP)]
    for f in kept:                      z.write('run/refs/' + f,   'refs/' + f)
    for f in os.listdir('run/gen'):     z.write('run/gen/' + f,    'gen/' + f)
    for f in os.listdir('run/inputs'):  z.write('run/inputs/' + f, 'inputs/' + f)
    for f in os.listdir('run/meta'):    z.write('run/meta/' + f,   'meta/' + f)
with zipfile.ZipFile(f'/content/{name}.zip') as z:
    assert z.testzip() is None, 'the zip is corrupt'
    names = z.namelist(); n_gen = sum(n.startswith('gen/') for n in names)
    for need in ('meta/prompts_v35.json', 'meta/run_v35.json', 'meta/cost_v35.json'):
        assert need in names, f'missing {need} from the zip'
    assert n_gen >= (len(ARMS) + 2) * want, f'only {n_gen} generated cells in the zip'
os.makedirs(os.path.join(BASE, 'v3_runs'), exist_ok=True)
shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', f'{name}.zip'))
print(f'{name}.zip · {len(names)} files ({n_gen} cells, {len(kept)} refs) · '
      f'{os.path.getsize(f"/content/{name}.zip") / 1e6:.0f} MB -> Drive v3_runs/')
try:
    from google.colab import files; files.download(f'/content/{name}.zip')
except Exception as e:
    print('download it from Drive instead:', e)